## K8sGPT

![](https://k8sgpt.ai/images/k8sgpt-logo.png)

Quelle: [K8sGPT](https://k8sgpt.ai/)

- - -

k8sgpt ist ein Werkzeug zur Analyse von Kubernetes-Clustern, das Fehlermeldungen und Konfigurationsprobleme erkennt und mit Hilfe von KI verständlich erklärt. 

Es unterstützt Administratoren dabei, Ursachen schneller zu identifizieren und passende Lösungsvorschläge zu erhalten. Damit kann k8sgpt als Einstieg in AIOps für Kubernetes betrachtet werden.

---

Es erstes Installieren wie das CLI

In [ ]:
%%bash
wget -nv https://github.com/k8sgpt-ai/k8sgpt/releases/latest/download/k8sgpt_Linux_x86_64.tar.gz
tar -xzf k8sgpt_Linux_x86_64.tar.gz
sudo mv k8sgpt /usr/local/bin/

---

### K8s Operator installieren 

Dieser Helm-Befehl installiert oder aktualisiert den `k8sgpt-operator` im Namespace `k8sgpt-operator-system` und aktiviert dabei zusätzlich die Prometheus-Integration über einen `ServiceMonitor` sowie ein Grafana-Dashboard im Namespace `opentelemetry`.


In [ ]:
%%bash
helm repo add k8sgpt https://charts.k8sgpt.ai
helm repo update
helm upgrade --install k8sgpt-operator k8sgpt/k8sgpt-operator \
  -n k8sgpt-operator-system \
  --create-namespace \
  --set serviceMonitor.enabled=true \
  --set serviceMonitor.namespace=opentelemetry \
  --set grafanaDashboard.enabled=true \
  --set grafanaDashboard.namespace=opentelemetry

Dieser Befehl erstellt eine `K8sGPT`-Custom-Resource im Namespace `k8sgpt-operator-system`, mit der der k8sgpt-Operator angewiesen wird, den Namespace `yaml` alle 60 Sekunden zu analysieren. Die KI-Auswertung ist dabei mit `enabled: false` deaktiviert; k8sgpt führt also die Kubernetes-Analyse aus, verwendet aber keinen OpenAI-Backend-Aufruf für zusätzliche Erklärungen.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: core.k8sgpt.ai/v1alpha1
kind: K8sGPT
metadata:
  name: k8sgpt
  namespace: k8sgpt-operator-system
spec:
  ai:
    enabled: false
    backend: openai
    model: gpt-4o-mini
  targetNamespace: yaml
  analysis:
    interval: 1m
  noCache: false
  version: v0.4.32
EOF

---

## Fehlerhaften Pod/Services starten

Diese absichtlich fehlerhafte Konfiguration erstellt im Namespace `yaml` einen Pod mit dem Label `app.kubernetes.io/name=webshop` und einen Service, der jedoch nach Pods mit dem Label `app.kubernetes.io/name=webshop2` sucht. Da der Selector des Service nicht zum Label des Pods passt, findet der Service keine passenden Endpoints und kann keinen Traffic an den Pod weiterleiten.


In [ ]:
%%bash
kubectl create namespace yaml
cat <<%EOF% | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  labels:
    app.kubernetes.io/name: webshop
  name: webshop
  namespace: yaml
spec:
  containers:
  - image: registry.gitlab.com/ch-mc-b/autoshop/shop:2.0.0
    name: webshop
%EOF%
cat <<%EOF% | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  labels:
    app.kubernetes.io/name: webshop
  name: webshop
  namespace: yaml
spec:
  ports:
  - port: 8080
    protocol: TCP
    targetPort: 8080
  selector:
    app.kubernetes.io/name: webshop2
  type: LoadBalancer
%EOF%

Der Befehl `k8sgpt analyze -n yaml` startet eine Analyse des Kubernetes-Namespaces `yaml` und prüft dort vorhandene Ressourcen auf typische Fehlkonfigurationen oder Betriebsprobleme. In diesem Beispiel sollte k8sgpt erkennen, dass der Service `webshop` keine Endpoints hat, weil sein Selector nicht zum Label des Pods passt.


In [ ]:
%%bash
k8sgpt analyze -n yaml     

Mit `k8sgpt analyze -n yaml -l german --explain` wird die Analyse des Namespaces `yaml` zusätzlich auf Deutsch erklärt; dafür ist eine konfigurierte OpenAI-Anmeldung beziehungsweise ein hinterlegter API-Zugang nötig, damit k8sgpt die gefundenen Probleme detailliert mit KI-Unterstützung ausformulieren kann.


In [ ]:
%%bash
source ~/data/env.py
cat ~/data/env.py
k8sgpt auth add --backend openai --model gpt-4o --password ${OPENAI_API_KEY}

In [ ]:
%%bash
k8sgpt analyze -n yaml -l german --explain

Mit `kubectl get results -A` werden die vom k8sgpt-Operator erzeugten Analyseergebnisse clusterweit angezeigt. Anschliessend liefert `kubectl describe -n k8sgpt-operator-system result/yamlwebshop` die Detailansicht zum gefundenen Resultat für den Service `webshop` im Namespace `yaml`, inklusive Fehlerbeschreibung und möglicher Lösungshinweise.


In [ ]:
%%bash
kubectl get results -A
kubectl describe -n k8sgpt-operator-system result/yamlwebshop

---

### AIOps

AIOps unterstützt den Betrieb von Kubernetes-Umgebungen, indem Betriebsdaten automatisch analysiert, Fehlerbilder erkannt und mögliche Ursachen abgeleitet werden. Ein typisches Beispiel ist ein Service, der keine Endpoints besitzt. In diesem Fall erwartet der Service Pods mit dem Label app.kubernetes.io/name=webshop2, findet jedoch keine passenden Pods. Dadurch kann der Service keinen Traffic an eine Anwendung weiterleiten.

Genau hier liegt die Aufgabe von AIOps: Es soll nicht nur melden, dass der Service keine Endpoints hat, sondern den Zusammenhang zwischen Service-Selector und Pod-Labels erkennen. Das System müsste feststellen, dass entweder das Label auf den Pods fehlt oder der Selector des Service falsch gesetzt ist. Anschliessend kann AIOps eine konkrete Handlungsempfehlung geben, zum Beispiel das fehlende Label auf den Pods zu ergänzen oder den Service-Selector an die tatsächlich vorhandenen Labels anzupassen.

**Leider ist diese weitergehende Analyse in k8sgpt aktuell noch nicht implementiert. k8sgpt erkennt zwar den Fehler, dass der Service keine Endpoints hat, leitet daraus aber noch nicht automatisch ab, welcher Pod betroffen ist oder welches Label konkret fehlt. Genau diese Korrelation zwischen Service, Selector und Pod-Labels wäre jedoch der nächste sinnvolle Schritt in Richtung AIOps.**



---

## Dashboard (Grafana) und Metricen scrappen

Das Grafana-Dashboard visualisiert die vom `k8sgpt-operator` erzeugten Analyse- und Zustandsdaten, damit erkannte Probleme nicht nur per CLI, sondern auch zentral im Monitoring sichtbar sind. 

Über den aktivierten `ServiceMonitor` können Prometheus beziehungsweise die OpenTelemetry-Monitoring-Umgebung die Metriken des Operators regelmässig scrapen. 

Dadurch lassen sich k8sgpt-Ergebnisse in bestehende Observability-Prozesse integrieren und zum Beispiel für Alarme, Dashboards oder Verlaufsauswertungen nutzen.


In [ ]:
%%bash
kubectl label servicemonitor \
  k8sgpt-operator-controller-manager-metrics-monitor \
  -n opentelemetry \
  release=prometheus \
  --overwrite

In [ ]:
%%bash
NAMESPACE="opentelemetry"
SERVER_IP="$(cat ~/data/server-ip 2>/dev/null || true)"
PROMETHEUS_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-prometheus -o=jsonpath='{.spec.ports[?(@.port==9090)].nodePort}')"
GRAFANA_PORT="$(kubectl -n "${NAMESPACE}" get svc prometheus-grafana -o=jsonpath='{.spec.ports[?(@.port==80)].nodePort}')"

echo "Grafana UI      : http://${SERVER_IP}:${GRAFANA_PORT}"
echo "Prometheus UI   : http://${SERVER_IP}:${PROMETHEUS_PORT}"

---

### Aufräumen


In [ ]:
%%bash
helm uninstall -n k8sgpt-operator-system k8sgpt-operator
kubectl delete ns yaml --wait=false
kubectl delete ns k8sgpt-operator-system --wait=false